# Vanilla LLM agent sandbox

Walkthrough of `prompt_manager` + `VanillaLLMAgent` on a small multi-asset episode.

**Prereqs:** `uv sync --extra llm`, Ollama running, and a pulled model (default `qwen3.5:2b`).
Cached prices help (`uv run alphaduel download -c configs/experiment/p5_genportfolio.yaml`).

In [1]:
import os
from pathlib import Path

# Notebook lives in notebooks/; make the project root the CWD so configs/cache resolve.
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    os.chdir(ROOT.parent)
print("cwd:", Path.cwd())

cwd: /home/steve77/Projects/alphaduel


## 1. Prompt manager

Load Jinja2 templates from `resources/prompts/` by key.

In [2]:
from alphaduel.prompts import prompt_manager

print("available:", prompt_manager.keys())

system = prompt_manager.get("base_agent_system")
print("--- system (first 400 chars) ---")
print(system.content[:400], "...")

user = prompt_manager.get("base_agent_user")
print("\n--- rendered user ---")
print(user.render(market_state="(example market brief)", symbols=["AAPL", "MSFT"]))

available: ['base_agent_system', 'base_agent_user']
--- system (first 400 chars) ---
You are an expert portfolio trader operating inside AlphaDuel.

## Goal
Each step you receive a structured market state. You must:
1. Reason in free-form natural language about what you see (momentum, mean reversion, risk, cash, costs).
2. Decide **integer share deltas** for tickers you want to trade.
3. End your reply with a single fenced JSON action block.

## Action rules
- Positive integer → * ...

--- rendered user ---
Here is the current market state for symbols: AAPL, MSFT.

(example market brief)

Write your analysis, then end with a fenced JSON actions block.



## 2. Build a real multi-asset panel + env

Uses the same pipeline as the CLI (`yfinance` cache → features → `MultiAssetGym`).

In [3]:
from alphaduel.config.loader import load_experiment_config
from alphaduel.config.schema import Secrets
from alphaduel.envs.multi_asset_gym import MultiAssetGym
from alphaduel.pipeline import build_panel

cfg = load_experiment_config("configs/experiment/p5_genportfolio.yaml")
# Keep the episode short for interactive demos.
cfg = cfg.model_copy(
    update={
        "data": cfg.data.model_copy(
            update={"symbols": ["AAPL", "MSFT", "GOOGL", "AMZN"]}
        ),
        "env": cfg.env.model_copy(update={"episode_length": 20, "random_start": False}),
    }
)

panel = build_panel(cfg, Secrets())
env = MultiAssetGym(panel, cfg.env, seed=cfg.seed)
obs, info = env.reset(seed=0)

print("symbols:", info["symbols"])
print("timestamp:", info["timestamp"])
print(f"equity={info['equity']:,.2f}  cash={info['cash']:,.2f}")
print("obs shape:", obs.shape)

symbols: ['AAPL', 'MSFT', 'GOOGL', 'AMZN']
timestamp: 2015-02-03 00:00:00+00:00
equity=100,000.00  cash=100,000.00
obs shape: (38,)


## 3. Preview the structured market brief

This is the string the LLM sees (before Jinja wraps it in the user prompt).

In [4]:
from alphaduel.agents.llm.state_format import format_market_state

brief = format_market_state(obs, info)
print(brief)

=== MARKET STATE ===
timestamp: 2015-02-03 00:00:00+00:00
equity: 100,000.00
cash: 100,000.00
n_assets: 4

=== POSITIONS ===
TICKER         SHARES        PRICE     WEIGHT
AAPL                0        26.25     0.00%
MSFT                0        35.30     0.00%
GOOGL               0        26.43     0.00%
AMZN                0        18.18     0.00%

=== FEATURES ===
TICKER          ret_1      mom_5      vol_5     mom_10     vol_10     mom_20     vol_20     rsi_14
AAPL           0.0002     0.0871     0.0271     0.0913     0.0249     0.1167     0.0224     0.6490
MSFT           0.0077    -0.0248     0.0298    -0.1033     0.0376    -0.1021     0.0283     0.3189
GOOGL          0.0021     0.0232     0.0247     0.0458     0.0231     0.0266     0.0180     0.6484
AMZN          -0.0025     0.1852     0.0553     0.2560     0.0416     0.2031     0.0333     0.8252

=== ACTION PROTOCOL ===
Decide share deltas for tickers you want to trade.
Positive = buy shares, negative = sell shares, omit = hold (

## 4. Run one LLM decision

Build the chat model with `LLMHandler.create(...)`, then inject it into `VanillaLLMAgent`. The agent parses a fenced `{"actions": {...}}` block and maps share deltas → env target weights.

Toggle `use_memory` to keep prior human/AI turns across steps.

In [5]:
from alphaduel.agents.llm import LLMHandler, VanillaLLMAgent

MODEL = "qwen3.5:2b"  # change if you pulled a different Ollama tag
USE_MEMORY = False

llm = LLMHandler.create(MODEL, provider="ollama", temperature=0.0, reasoning=False)
agent = VanillaLLMAgent(llm=llm, use_memory=USE_MEMORY, max_memory_turns=6)
agent.reset()

action = agent.act(obs, info)

print("parse_ok:", agent.last_parse_ok)
print("actions:", agent.last_actions)
print("target weights:", action)
print("\n--- rationale (raw LLM reply) ---\n")
print(agent.last_thoughts)

parse_ok: True
actions: {'MSFT': -2}
target weights: [0. 0. 0. 0.]

--- rationale (raw LLM reply) ---

The market state shows AAPL and GOOGL in strong uptrends (positive ret_1, positive mom_5), while MSFT is showing mean reversion signals (-0.0248 momentum) despite being priced at 35.30 vs the ~26-27 range of its peers. AMZN has a high vol_5 but negative ret_1 and low mom_5, suggesting it might be overbought or in a dip relative to recent performance compared to AAPL/GOOGL.

I see MSFT is currently underperforming the 20-day momentum mean (ret_1 = -0.0248) while holding significant weight at 35% of my portfolio, which creates an immediate risk-reward imbalance if I want to rebalance for a short horizon. AAPL and GOOGL are both showing positive momentum but have lower weights; adding more here would increase volatility exposure without the mean reversion safety net MSFT offers.

I will trim some MSFT positions to reduce my weight in the underperforming asset, while holding all other tic

## 5. Step the env and run a short loop

Execute the action, then ask the agent again for a few more bars.

In [6]:
import pandas as pd

rows = []
obs, info = env.reset(seed=0)
agent.reset()

for t in range(5):
    action = agent.act(obs, info)
    rows.append(
        {
            "t": t,
            "timestamp": str(info["timestamp"]),
            "equity": info["equity"],
            "parse_ok": agent.last_parse_ok,
            "actions": dict(agent.last_actions),
            "weights": action.round(4).tolist(),
        }
    )
    obs, reward, terminated, truncated, info = env.step(action)
    rows[-1]["reward"] = reward
    if terminated or truncated:
        break

pd.DataFrame(rows)

,t,timestamp,equity,parse_ok,actions,weights,reward
0,0,2015-02-03 00:00:00+00:00,100000.000000,True,{'MSFT': -2},"[0.0, 0.0, 0.0, 0.0]",0.000000
1,1,2015-02-04 00:00:00+00:00,100000.000000,True,{'AAPL': 10},"[0.0026000000070780516, 0.0, 0.0, 0.0]",-0.000003
2,2,2015-02-05 00:00:00+00:00,99999.723071,True,{'AAPL': 1},"[0.0027000000700354576, 0.0, 0.0, 0.0]",-0.000020
3,3,2015-02-06 00:00:00+00:00,99997.703612,True,"{'MSFT': -1, 'GOOGL': 2}","[0.002400000113993883, 0.0, 0.0005000000237487...",0.000014
4,4,2015-02-09 00:00:00+00:00,99999.081015,True,"{'AAPL': 2, 'MSFT': -1}","[0.002899999963119626, 0.0, 0.0005000000237487...",0.000057


## 6. Offline parse check (no Ollama)

Useful to verify the JSON extractor without calling a model.

In [ ]:
from alphaduel.agents.llm.parse import parse_llm_response

samples = [
    '''AAPL looks strong; adding a clip.
```json
{"actions": {"AAPL": 3, "MSFT": -1}}
```''',
    "I might buy something later but I won't emit JSON.",
    '''No edge today.
```json
{"actions": {}}
```''',
]

for raw in samples:
    parsed = parse_llm_response(raw)
    print(f"parse_ok={parsed.parse_ok!s:5}  actions={parsed.actions}")